In [1]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/wholesale.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
pd.set_option("display.max_columns", None)
df.head()

,V1,V2,V3,V4,V5,V6,V7,Channel,Region
0,3.0,12669.0,9656.0,7561.0,214.0,2674.0,1338.0,2,3
1,3.0,7057.0,9810.0,9568.0,1762.0,3293.0,1776.0,2,3
2,3.0,6353.0,8808.0,7684.0,2405.0,3516.0,7844.0,2,3
3,3.0,13265.0,1196.0,4221.0,6404.0,507.0,1788.0,1,3
4,3.0,22615.0,5410.0,7198.0,3915.0,1777.0,5185.0,2,3


In [3]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает > 90% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если константных значений или пропусков больше 95% -> к удалению
        if max_value_ratio > 0.95:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df)

In [5]:
df['Channel'] = df['Channel'].map({'2' : 0, '1' : 1 })
y = df["Channel"]
X = df.drop(["Channel"], axis=1)

In [ ]:
import time
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from k_fold_encoder import KFoldTargetEncoder
from k_fold_glmm_encoder import KFoldGLMMEncoder

cat_cols = ["Region"]

encoders = {
   'Leave-one-out': lambda: ce.LeaveOneOutEncoder(cols=cat_cols),
   'K-fold':        lambda: KFoldTargetEncoder(cols=cat_cols),
   "CatBoost":      lambda: ce.CatBoostEncoder(cols=cat_cols),
   "CV-GLMM":       lambda: KFoldGLMMEncoder(cols=cat_cols),
   "WOE":           lambda: ce.WOEEncoder(cols=cat_cols),
   "JS":            lambda: ce.JamesSteinEncoder(cols=cat_cols),
   "MT":            lambda: ce.TargetEncoder(cols=cat_cols),
   'GLMM':          lambda: ce.GLMMEncoder(cols=cat_cols),
   'M-Estimate':    lambda: ce.MEstimateEncoder(cols=cat_cols),
   "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
   "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
   "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
   "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
   "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
   "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
   "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols),
   "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough")
}

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():
    for encoder_name, encoder in encoders.items():
        
        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            enc_instance = encoder()  # Вызываем lambda, получая объект кодировщика
            X_train_encoded = enc_instance.fit_transform(X_train, y_train)
            X_test_encoded = enc_instance.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Empirical",
                    "dataset": "Wholesale",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Empirical, Wholesale, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_wholesale_empirical.csv", index=True)
print("Done!")




Empirical, Wholesale, LogReg, Leave-one-out, 1, 1, 0.9333722969023963, 0.947101843961932, 0.8977272727272727, 0.926829268292683, 0.890625, 0.9661016949152542, 0.03605222702026367
Empirical, Wholesale, LogReg, Leave-one-out, 1, 2, 0.9421390999415546, 0.9344708101866575, 0.9090909090909091, 0.9344262295081968, 0.9047619047619048, 0.9661016949152542, 0.03341031074523926
Empirical, Wholesale, LogReg, Leave-one-out, 1, 3, 0.956547619047619, 0.979975460613769, 0.8863636363636364, 0.9137931034482759, 0.9464285714285714, 0.8833333333333333, 0.03456401824951172
Empirical, Wholesale, LogReg, Leave-one-out, 1, 4, 0.9630952380952382, 0.9774499490117773, 0.9204545454545454, 0.943089430894309, 0.9206349206349206, 0.9666666666666667, 0.032036781311035156
Empirical, Wholesale, LogReg, Leave-one-out, 1, 5, 0.9779761904761904, 0.9914598442273161, 0.9204545454545454, 0.943089430894309, 0.9206349206349206, 0.9666666666666667, 0.024769306182861328
Empirical, Wholesale, LogReg, Leave-one-out, 2, 1, 0.978375

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, Wholesale, LightGBM, MinHash, 1, 1, 0.9538281706604326, 0.9700507457761842, 0.9204545454545454, 0.9401709401709402, 0.9482758620689655, 0.9322033898305084, 0.15140652656555176
Empirical, Wholesale, LightGBM, MinHash, 1, 2, 0.9421390999415546, 0.9654944741427732, 0.9204545454545454, 0.9401709401709402, 0.9482758620689655, 0.9322033898305084, 0.11747074127197266


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, Wholesale, LightGBM, MinHash, 1, 3, 0.9488095238095238, 0.9786227045613844, 0.8977272727272727, 0.9243697478991597, 0.9322033898305084, 0.9166666666666666, 0.12407708168029785
Empirical, Wholesale, LightGBM, MinHash, 1, 4, 0.9744047619047619, 0.9864124704283463, 0.9318181818181818, 0.9508196721311475, 0.9354838709677419, 0.9666666666666667, 0.11715102195739746


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


... (33 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have v

Empirical, Wholesale, LightGBM, Similarity, 5, 3, 0.9452380952380952, 0.9635053775237512, 0.9090909090909091, 0.9354838709677419, 0.90625, 0.9666666666666667, 0.0969388484954834
Empirical, Wholesale, LightGBM, Similarity, 5, 4, 0.9541666666666666, 0.981149658300258, 0.9090909090909091, 0.9310344827586207, 0.9642857142857143, 0.9, 0.10299396514892578
Empirical, Wholesale, LightGBM, Similarity, 5, 5, 0.9708333333333333, 0.9821999477032065, 0.9431818181818182, 0.959349593495935, 0.9365079365079365, 0.9833333333333333, 0.09537577629089355


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, Wholesale, CatBoost, Leave-one-out, 1, 1, 0.9374634716540035, 0.9588864431671941, 0.6818181818181818, 0.8082191780821918, 0.6781609195402298, 1.0, 1.6346564292907715
Empirical, Wholesale, CatBoost, Leave-one-out, 1, 2, 0.9637638807714787, 0.9797285623604024, 0.7159090909090909, 0.8251748251748252, 0.7023809523809523, 1.0, 1.5290560722351074
Empirical, Wholesale, CatBoost, Leave-one-out, 1, 3, 0.9446428571428571, 0.9715921187095936, 0.7272727272727273, 0.8333333333333334, 0.7142857142857143, 1.0, 1.5139400959014893
Empirical, Wholesale, CatBoost, Leave-one-out, 1, 4, 0.9285714285714285, 0.9692344917739195, 0.7159090909090909, 0.8275862068965517, 0.7058823529411765, 1.0, 1.4626827239990234
Empirical, Wholesale, CatBoost, Leave-one-out, 1, 5, 0.9589285714285715, 0.9842748872189067, 0.7272727272727273, 0.8333333333333334, 0.7142857142857143, 1.0, 1.4353840351104736
Empirical, Wholesale, CatBoost, Leave-one-out, 2, 1, 0.9549970777323203, 0.9786557919891268, 0.7159090909090909, 0.